# ⚡ AlgoHub Week 8 Capstone: End-to-End Data Pipeline
### Dataset: Real IBM Telco Customer Churn
**Objective:** Build a production-grade, reusable scikit-learn preprocessing and machine learning pipeline using custom transformers, automated EDA, joblib serialization, and inference verification.


In [1]:
import sys
import os
from pathlib import Path

# Ensure project root is in python path
root_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from src.config import RAW_DATA_PATH, PIPELINE_SAVE_PATH, PREPROCESSOR_SAVE_PATH, MODEL_METRICS_PATH, RANDOM_STATE
from src.custom_transformers import (
    IDColumnDropper, DataTypeCoercer, MissingValueHandler, OutlierCapper,
    FeatureEngTransformer, CategoricalEncoder, FeatureScaler
)
from src.utils import load_raw_data, ensure_directories

print("Environment setup completed successfully.")


Environment setup completed successfully.


## 1. Real Dataset Loading & Quality Analysis
Inspect raw data dimensions, column types, space string missing values in `TotalCharges`, and target class distribution.


In [2]:
df_raw = load_raw_data()
print("Raw Dataset Shape:", df_raw.shape)
print("\nTarget Churn Distribution:")
print(df_raw['Churn'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))

space_count = (df_raw['TotalCharges'].astype(str).str.strip() == '').sum()
print(f"\nDetected {space_count} whitespace missing value records in TotalCharges column.")
df_raw.head(5)


Raw Dataset Shape: (7043, 21)

Target Churn Distribution:
Churn
No     73.46%
Yes    26.54%
Name: proportion, dtype: object

Detected 11 whitespace missing value records in TotalCharges column.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Verification of Custom Scikit-Learn Transformers
Test individual custom transformers (`BaseEstimator` and `TransformerMixin`).


In [3]:
# 1. IDColumnDropper & DataTypeCoercer
dropper = IDColumnDropper(id_cols=["customerID"])
coercer = DataTypeCoercer(target_cols=["TotalCharges"])

df_step1 = dropper.fit_transform(df_raw)
df_step2 = coercer.fit_transform(df_step1)

print("customerID in columns after dropper:", "customerID" in df_step2.columns)
print("TotalCharges dtype after coercer:", df_step2["TotalCharges"].dtype)
print("TotalCharges NaN count after coercer:", df_step2["TotalCharges"].isnull().sum())


customerID in columns after dropper: False
TotalCharges dtype after coercer: float64
TotalCharges NaN count after coercer: 11


In [4]:
# 2. MissingValueHandler & FeatureEngTransformer
imputer = MissingValueHandler(numeric_cols=["tenure", "MonthlyCharges", "TotalCharges"], categorical_cols=["Contract"])
df_step3 = imputer.fit_transform(df_step2)

fe = FeatureEngTransformer()
df_step4 = fe.fit_transform(df_step3)

print("TotalCharges NaNs after imputer:", df_step4["TotalCharges"].isnull().sum())
print("Newly Engineered Features:")
df_step4[["tenure", "TenureYears", "AvgMonthlyCostPerTenure", "HasAddonsCount", "IsLongTermContract", "ElectronicPayment"]].head(5)


TotalCharges NaNs after imputer: 0
Newly Engineered Features:


,tenure,TenureYears,AvgMonthlyCostPerTenure,HasAddonsCount,IsLongTermContract,ElectronicPayment
0,1,0.083333,14.925000,1,0,1
1,34,2.833333,53.985714,2,1,0
2,2,0.166667,36.050000,2,0,0
3,45,3.750000,40.016304,3,1,0
4,2,0.166667,50.550000,0,0,1


## 3. Full Preprocessing Pipeline Construction
Combine custom transformers into a unified `Pipeline` and transform feature set.


In [5]:
from src.pipeline_builder import build_preprocessor

preprocessor = build_preprocessor()

X = df_raw.drop(columns=["Churn"])
y = (df_raw["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f"Transformed Feature Matrix Shape: {X_train_trans.shape} (51 engineered & one-hot encoded features)")


Train set shape: (5634, 20), Test set shape: (1409, 20)


Transformed Feature Matrix Shape: (5634, 51) (51 engineered & one-hot encoded features)


## 4. Machine Learning Model Training & Real Evaluation
Train candidate classifiers (Logistic Regression, Random Forest, Gradient Boosting) on real data and calculate genuine metrics.


In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

eval_results = {}
for name, clf in models.items():
    clf.fit(X_train_trans, y_train)
    preds = clf.predict(X_test_trans)
    probas = clf.predict_proba(X_test_trans)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, probas)
    
    eval_results[name] = {
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "ROC-AUC": round(roc, 4),
    }

metrics_df = pd.DataFrame(eval_results).T
print("=== Model Benchmarking Comparison Table ===")
metrics_df


=== Model Benchmarking Comparison Table ===


,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.8105,0.6814,0.5374,0.6009,0.8471
Random Forest,0.7913,0.6342,0.5053,0.5625,0.8235
Gradient Boosting,0.8048,0.6737,0.5134,0.5827,0.8432


In [7]:
# Display Classification Report for Winning Model (Logistic Regression)
best_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
best_clf.fit(X_train_trans, y_train)
y_pred_best = best_clf.predict(X_test_trans)

print("=== Winning Model Classification Report (Logistic Regression) ===")
print(classification_report(y_test, y_pred_best, target_names=["No Churn (0)", "Churn (1)"]))


=== Winning Model Classification Report (Logistic Regression) ===
              precision    recall  f1-score   support

No Churn (0)       0.84      0.91      0.88      1035
   Churn (1)       0.68      0.54      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## 5. Joblib Serialization & Inference Verification
Save full pipeline to disk (`models/pipeline.joblib`), reload it, and verify inference on new raw customer records.


In [8]:
# Assembly of complete Pipeline (Preprocessor + Classifier)
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_clf)
])

ensure_directories()
joblib.dump(full_pipeline, PIPELINE_SAVE_PATH)
print("Saved complete pipeline to:", PIPELINE_SAVE_PATH)

# Reload Pipeline from disk
loaded_pipeline = joblib.load(PIPELINE_SAVE_PATH)

# Sample Raw Input Record
sample_raw_customer = pd.DataFrame([{
    "customerID": "NOTEBOOK-INFERENCE-999",
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.5,
    "TotalCharges": "85.5"
}])

prediction = loaded_pipeline.predict(sample_raw_customer)[0]
probability = loaded_pipeline.predict_proba(sample_raw_customer)[0][1]

print("\n=== Reloaded Joblib Pipeline Real Inference Result ===")
print(f"Customer ID: {sample_raw_customer['customerID'].iloc[0]}")
print(f"Predicted Churn: {'Yes (1)' if prediction == 1 else 'No (0)'}")
print(f"Churn Probability: {probability:.4f}")
print(f"Assessed Risk Level: {'High' if probability >= 0.65 else ('Medium' if probability >= 0.35 else 'Low')}")


Saved complete pipeline to: C:\Users\FAIZAN COMPUTERS\Desktop\AlgoHub_Week8_End_to_End_Data_Pipeline\models\pipeline.joblib



=== Reloaded Joblib Pipeline Real Inference Result ===
Customer ID: NOTEBOOK-INFERENCE-999
Predicted Churn: Yes (1)
Churn Probability: 0.8525
Assessed Risk Level: High
